# LG HelloDoctor — B팀 의료 LLM 파인튜닝
> 데이터 로드 → LoRA 파인튜닝 → 의도분류 → Entity추출
> → Groq fallback → 평가 → 다중 턴 대화

## Step 1 — 라이브러리 설치

In [1]:
!pip install unsloth trl datasets bitsandbytes groq -q
print('설치 완료!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 9.5 MB/s eta 0:00:00
설치 완료!


## Step 2 — Google Drive 연결

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/model', exist_ok=True)
os.makedirs('/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf', exist_ok=True)
print('Drive 연결 완료!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive 연결 완료!


## Step 3 — 데이터 로드

In [3]:

import gdown

gdown.download(
    'https://drive.google.com/uc?id=12QMAcn63E503HUMpVwiftj9ClycmkNvJ',
    '/content/train.jsonl',
    quiet=False
)

Downloading...
From: https://drive.google.com/uc?id=12QMAcn63E503HUMpVwiftj9ClycmkNvJ
To: /content/train.jsonl
100%|██████████| 477k/477k [00:00<00:00, 120MB/s]


'/content/train.jsonl'

In [4]:
from datasets import load_dataset

# 학습 데이터
dataset = load_dataset(
    'json',
    data_files='/content/drive/MyDrive/LG_HelloDoctor/LLM/data/train.jsonl',
    split='train'
)

# 검증 데이터 추가
eval_dataset = load_dataset(
    'json',
    data_files='/content/drive/MyDrive/LG_HelloDoctor/LLM/data/validation.jsonl',
    split='train'
)

print(f'학습 데이터: {len(dataset)}개')
print(f'검증 데이터: {len(eval_dataset)}개')

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

학습 데이터: 7200개
검증 데이터: 700개


In [ ]:
# from datasets import load_dataset

# dataset = load_dataset(
#     'json',
#     data_files='/content/drive/MyDrive/LG_HelloDoctor/LLM/data/00_all_medical_train.jsonl',
#     split='train'
# )
# print(result.stdout)
# print(f'데이터 로드 완료: {len(dataset)}개')
# print('샘플 확인:')
# print(dataset[0])


## Step 4 — 모델 로드 (4bit 양자화)

In [ ]:
!pip install unsloth

In [5]:
from unsloth import FastLanguageModel

# ⚠️ 변수명을 llm_model 로 고정 (Whisper model 변수와 충돌 방지)
llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/llama-3.2-3b-instruct',
    max_seq_length=512,
    load_in_4bit=True,
)
print('LLaMA 모델 로드 완료!')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


LLaMA 모델 로드 완료!


## Step 5 — LoRA 설정

In [6]:
# LoRA 설정
llm_model = FastLanguageModel.get_peft_model(
    llm_model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.15,       # ← 0.01 → 0.15 (과적합 방지)
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
)
print('LoRA 설정 완료!')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.15.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA 설정 완료!


## Step 6 — 프롬프트 포맷

In [7]:
import re

# ============================================================
# [개선 1] 의도 라벨 자동 생성 함수
# ============================================================
EMERGENCY_KW_TRAIN = ['숨이 안', '숨을 못', '의식이 없', '쓰러', '피를 토', '마비', '경련', '발작', '갑자기 안 보', '말이 안 나', '입이 돌아', '한쪽이 마비', '심한 출혈', '호흡 곤란']
HOSPITAL_KW_TRAIN  = ['병원', '어디예요', '어디 있', '위치', '운영시간', '진료시간', '몇 시까지', '한의원', '치과', '산부인과']
MEDICATION_KW_TRAIN = ['약', '복용', '먹어도', '부작용', '용량', '처방']

def auto_label_intent(text: str) -> str:
    if any(kw in text for kw in EMERGENCY_KW_TRAIN):
        return 'emergency'
    if any(kw in text for kw in HOSPITAL_KW_TRAIN):
        return 'hospital_search'
    if any(kw in text for kw in MEDICATION_KW_TRAIN):
        return 'medication_info'
    return 'symptom_inquiry'

def format_prompt(example):
    intent = auto_label_intent(example['instruction'])
    return {
        'text': (
            f"### 의도분류\n"
            f"문장: {example['instruction']}\n"
            f"의도: {intent}\n\n"
            f"### 질문:\n{example['instruction']}\n\n"
            f"### 답변:\n{example['output']}<|end_of_text|>"
        )
    }

dataset = dataset.map(format_prompt)
print('프롬프트 포맷 완료! (의도 라벨 포함)')
print('샘플:', dataset[0]['text'][:200])


Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

프롬프트 포맷 완료! (의도 라벨 포함)
샘플: ### 의도분류
문장: [1턴] 설명하기가 좀 그런데 은근히 잇몸이 욱신거려요좀 봐주세요.
[AI] 씹을 때만 더 아픈가요?
[2턴] 가만히 있어도 아파요
의도: symptom_inquiry

### 질문:
[1턴] 설명하기가 좀 그런데 은근히 잇몸이 욱신거려요좀 봐주세요.
[AI] 씹을 때만 더 아픈가요?
[2턴] 가만히 있어도 아파요

### 답변:
치


## Step 7 — 학습

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# validation.jsonl → text 형식으로 변환
eval_texts = []
for item in eval_dataset:
    text = f"### 질문:\n{item['instruction']}\n\n### 답변:\n{item['output']}<|end_of_text|>"
    eval_texts.append({'text': text})

eval_dataset_converted = Dataset.from_list(eval_texts)
print(f'변환된 검증 데이터: {len(eval_dataset_converted)}개')

def formatting_func(example):
    if 'text' in example and example['text']:
        return [example['text']]
    return [f"### 질문:\n{example['instruction']}\n\n### 답변:\n{example['output']}<|end_of_text|>"]

trainer = SFTTrainer(
    model=llm_model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset_converted,
    formatting_func=formatting_func,
    max_seq_length=1024,
    args=TrainingArguments(
        output_dir='/content/drive/MyDrive/LG_HelloDoctor/LLM/checkpoints',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_steps=100,          # ← 50에서 100으로 변경 (중복 X)
        learning_rate=1e-4,        # ← 2e-4에서 변경
        weight_decay=0.01,         # ← 새로 추가
        lr_scheduler_type="cosine", # ← 새로 추가
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_strategy='epoch',
        eval_strategy='epoch',
        load_best_model_at_end=True,
        gradient_checkpointing=False,
    )
)
trainer.train()
print('학습 완료!')

변환된 검증 데이터: 700개


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/7200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/700 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,200 | Num Epochs = 3 | Total steps = 1,350
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,0.159765,1.247131
2,0.140478,1.298484
3,0.133693,1.325934


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

학습 완료!


In [10]:
# 학습 로그 확인
log_history = trainer.state.log_history

print('=' * 45)
print('Epoch | Train Loss | Eval Loss | 상태')
print('=' * 45)

for log in log_history:
    if 'eval_loss' in log:
        epoch      = log.get('epoch', '')
        eval_loss  = log.get('eval_loss', 0)
        train_loss = None

        # 같은 에폭 train_loss 찾기
        for l in log_history:
            if 'loss' in l and abs(l.get('epoch', -1) - epoch) < 0.1:
                train_loss = l['loss']
                break

        if train_loss:
            diff = eval_loss - train_loss
            status = '✅ 정상' if diff < 0.3 else '⚠️ 과적합 의심'
            print(f'  {epoch:.0f}  |   {train_loss:.4f}   |   {eval_loss:.4f}  | {status}')
        else:
            print(f'  {epoch:.0f}  |     -     |   {eval_loss:.4f}  |')

print('=' * 45)
print()
print('판단 기준:')
print('  eval_loss 계속 감소  → 정상 학습')
print('  eval_loss 증가       → 과적합 발생')
print('  (load_best_model_at_end=True로 최적 모델 자동 저장됨)')

Epoch | Train Loss | Eval Loss | 상태
  1  |   0.1570   |   1.2471  | ⚠️ 과적합 의심
  2  |   0.1405   |   1.2985  | ⚠️ 과적합 의심
  3  |   0.1369   |   1.3259  | ⚠️ 과적합 의심

판단 기준:
  eval_loss 계속 감소  → 정상 학습
  eval_loss 증가       → 과적합 발생
  (load_best_model_at_end=True로 최적 모델 자동 저장됨)


## Step 8 — 모델 저장

In [11]:
llm_model.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor/LLM/model')
tokenizer.save_pretrained('/content/drive/MyDrive/LG_HelloDoctor/LLM/model')
print('모델 저장 완료!')


모델 저장 완료!


## Step 9 — GGUF 변환

In [12]:
llm_model.save_pretrained_gguf(
    '/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf',
    tokenizer,
    quantization_method='q4_k_m'
)
print('GGUF 변환 완료!')


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 1918.71it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:55<00:00, 117.51s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/LG_HelloDoctor/LLM/gguf_gguf/Modelfile
GGUF 변환 완료!


In [13]:
# 1. 설치
!pip install unsloth groq -q

# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')

# 3. 저장된 모델 불러오기
from unsloth import FastLanguageModel

llm_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='/content/drive/MyDrive/LG_HelloDoctor/LLM/model',
    max_seq_length=512,
    load_in_4bit=True,
)
print('모델 불러오기 완료!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


모델 불러오기 완료!


## Step 10 — 의도 분류기

In [14]:
import re

VALID_INTENTS = ['symptom_inquiry', 'hospital_search', 'medication_info', 'emergency']

def parse_intent_from_output(result: str) -> str:
    if not result:
        return 'symptom_inquiry'
    result = result.strip()
    match = re.search(r'의도\s*:\s*([A-Za-z_]+)', result)
    if match:
        pred = match.group(1).strip()
        if pred in VALID_INTENTS:
            return pred
    for intent in VALID_INTENTS:
        if intent in result:
            return intent
    tokens = result.split()
    if tokens:
        first = tokens[0].strip()
        if first in VALID_INTENTS:
            return first
    return 'symptom_inquiry'

# ── 응급 키워드 + 패턴 ──
EMERGENCY_KEYWORDS = [
    # 호흡
    '숨이 안 쉬', '숨을 못 쉬', '호흡이 안 돼', '숨이 막혀', '질식',
    '숨이 끊어',
    # 의식
    '의식이 없', '의식 잃', '기절했', '쓰러졌', '쓰러질 것 같',
    '정신을 잃',
    # 가슴 (구체적 표현만)
    '가슴이 너무 아프고 숨', '가슴을 쥐어짜', '가슴이 쥐어', '심장이 멈추',
    # 마비/신경
    '한쪽이 마비', '마비됐', '감각이 없어', '말이 안 나',
    '말이 어눌', '입이 돌아', '갑자기 안 보여', '갑자기 못 움직',
    '한쪽 팔이 갑자기 안 움직',
    # 출혈
    '피를 토', '출혈이 멈추지 않', '피가 안 멈춰', '대량 출혈',
    # 기타 응급
    '경련', '발작', '심한 복통이 갑자기',
    '갑자기 쓰러', '쓰러졌어요',
    #
    '화상을 심하게','심장이 갑자기','뼈가 부러진','대량 출혈',
]

EMERGENCY_PATTERNS = [
    r'숨.{0,5}(못|안|힘|막혀|차)',
    r'가슴.{0,8}(너무|심하|쥐어).{0,10}숨',
    r'(마비|감각).{0,5}(없|안)',
    r'갑자기.{0,10}(못|안|없|쓰러)',
    r'(극심한|참을\s?수\s?없는).{0,5}(통증|고통)',
    r'(한쪽|왼쪽|오른쪽).{0,5}(마비|힘이\s?없|안\s?움직)',
    r'피.{0,3}(토|멈추지|멈춰)',
    # ── 추가 ──
    r'갑자기.{0,10}(심하|너무).{0,10}(아파|통증|고통)',   # 갑자기 + 심하게 아파
    r'(심하게|극심하게).{0,5}(입었|다쳤|다침)',            # 화상·외상
    r'심장.{0,5}(갑자기|너무).{0,5}(빠르|뛰)',            # 심장 빠르게 뜀
    r'(뼈|골절).{0,5}(부러|골절)',                         # 골절
]

def is_emergency(text: str) -> bool:
    for kw in EMERGENCY_KEYWORDS:
        if kw in text:
            return True
    for pattern in EMERGENCY_PATTERNS:
        if re.search(pattern, text):
            return True
    return False

# ── 복약 키워드 확장 ──
MEDICATION_KW = [
    '약', '복용', '먹어도', '부작용', '용량', '처방', '먹어야',
    '항생제', '혈압약', '당뇨약', '진통제', '수면제', '고지혈증약',
    '끊어도', '끊으면', '먹고 나서', '먹기 전', '먹은 후',
    '복약', '영양제', '비타민', '소화제', '아스피린', '오메가',
    '약이 남', '약을 잊', '약을 잘못', '약이 너무 많',
]

# ── 병원 검색 키워드 확장 ──
HOSPITAL_KW = [
    '병원', '어디예요', '어디 있', '어디에', '근처', '위치',
    '진료시간', '몇 시', '알려주세요', '찾아주세요', '찾아줘',
    '운영시간', '예약', '검진', '진료해요', '문 열', '응급실',
    '한의원', '약국 어디', '보건소',
]

INTENT_PROMPT = """당신은 의료 AI 분류기입니다.
아래 문장을 읽고 의도를 4가지 중 하나로만 답하세요. 반드시 영어 키워드만 출력하세요.

[분류 기준]
- symptom_inquiry : 몸이 아프거나 불편한 증상을 말하는 경우
  예) 무릎이 아파요, 기침이 나요, 열이 나요, 머리가 아파요

- hospital_search : 병원 위치, 근처 병원, 진료 시간을 묻는 경우
  예) 가까운 내과 어디예요, 병원 있나요, 오늘 진료해요, 찾아주세요

- medication_info : 약 복용법, 약 정보, 약 부작용을 묻는 경우
  예) 혈압약 먹어도 되나요, 항생제 끊어도 되나요, 약 같이 먹어도 되나요

- emergency : 즉시 119가 필요한 위험한 상황
  예) 숨을 못 쉬어요, 의식을 잃었어요, 쓰러질 것 같아요, 마비됐어요

문장: {text}
의도:"""

def classify_intent(text: str) -> dict:
    # 멀티턴 전체 텍스트 + 마지막 발화
    if '[2턴]' in text:
        last_turn = text.split('[2턴]')[-1].strip()
        # 2턴에 부정 표현 있으면 응급 완화
        negation = ['괜찮', '아니요', '오래됐', '없어요', '그냥']
        if any(neg in last_turn for neg in negation):
            check_emergency = False
        else:
            check_emergency = True
    else:
        last_turn = text.strip()
        check_emergency = True

    full_text = text + ' ' + last_turn

    # 1순위: 응급 (부정 없을 때만)
    if check_emergency and is_emergency(full_text):
        return {'intent': 'emergency', 'confidence': 0.99, 'method': 'keyword'}

    # 2순위: 복약
    if any(kw in full_text for kw in MEDICATION_KW):
        return {'intent': 'medication_info', 'confidence': 0.90, 'method': 'keyword'}

    # 3순위: 병원 검색 (전체 텍스트로 체크)
    if any(kw in full_text for kw in HOSPITAL_KW):
        return {'intent': 'hospital_search', 'confidence': 0.90, 'method': 'keyword'}


    # 4순위: Groq로 분류 (model 대신)
    try:
        response = groq_client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[
                {'role': 'system', 'content': '의도를 symptom_inquiry / hospital_search / medication_info / emergency 중 하나로만 답하세요. 영어로만 출력하세요.'},
                {'role': 'user', 'content': f'문장: {last_turn}\n의도:'}
            ],
            max_tokens=10,
            temperature=0.0,
        )
        result = response.choices[0].message.content.strip()
        intent = parse_intent_from_output(result)
        return {'intent': intent, 'confidence': 0.85, 'method': 'groq'}
    except Exception as e:
        print(f'Groq 오류: {e}')
        return {'intent': 'symptom_inquiry', 'confidence': 0.5, 'method': 'fallback'}

## Step 11 — Entity 추출 (증상·부위·위치)

In [15]:
import json

BODY_PARTS = [
    '무릎', '허리', '어깨', '팔', '다리', '발', '손', '목',
    '머리', '눈', '귀', '코', '입', '치아', '잇몸', '가슴',
    '배', '위', '심장', '폐', '피부', '발목', '손목', '골반'
]

ENTITY_PROMPT = """다음 문장에서 증상과 신체 부위를 추출해서 JSON으로만 답하세요.
형식: {{"symptom": "증상", "body_part": "신체부위", "location": null}}
없으면 null로 표시하세요.

문장: {text}
JSON:"""

def extract_entities(text: str) -> dict:
    # 키워드 기반 빠른 추출
    found_parts = [p for p in BODY_PARTS if p in text]
    body_part = found_parts[0] if found_parts else None

    # LLM으로 정확한 추출
    prompt = ENTITY_PROMPT.format(text=text)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to('cuda')

    outputs = llm_model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=60,
        temperature=0.1,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_str = result.split('JSON:')[-1].strip()

    try:
        json_match = re.search(r'\{.*?\}', json_str, re.DOTALL)
        entities = json.loads(json_match.group()) if json_match else None
    except:
        entities = None

    if not entities:
        entities = {'symptom': None, 'body_part': body_part, 'location': None}

    return entities


# 테스트
test_cases = [
    '무릎이 너무 아파요. 어디 가야 해요?',
    '허리가 끊어질 것 같아요',
    '혈압약이랑 감기약 같이 먹어도 되나요?',
]

print('=== Entity 추출 테스트 ===')
for text in test_cases:
    entities = extract_entities(text)
    print(f'입력: {text}')
    print(f'결과: {entities}')
    print('-' * 40)

Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Entity 추출 테스트 ===


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

입력: 무릎이 너무 아파요. 어디 가야 해요?
결과: {'symptom': '관절이나 근육 문제', 'body_part': '관절', 'location': '외상이 심한 편이면 응급실, 외상이 심한 편이면 응급실을 먼저 이용해 주세요.'}
----------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 허리가 끊어질 것 같아요
결과: {'symptom': '허리나 목이나 기침이 있어요', 'body_part': '허리', 'location': None}
----------------------------------------
입력: 혈압약이랑 감기약 같이 먹어도 되나요?
결과: {'symptom': None, 'body_part': None, 'location': None}
----------------------------------------


## Step 12 — Groq Fallback 연동

In [16]:
from groq import Groq
from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_API_KEY = userdata.get('GROQ_API_KEY').strip()  # strip()으로 공백 제거
groq_client = Groq(api_key=GROQ_API_KEY)
SYSTEM_PROMPT = """당신은 노인 환자를 위한 의료 안내 AI 헬로비입니다.
반드시 아래 규칙을 지키세요:
- 3문장 이내로 답하세요
- 쉬운 말로 부드럽게 답하세요
- 존댓말을 사용하세요
- 질환을 단정하지 마세요
- 응급 상황이면 119를 먼저 안내하세요
- 금지 단어: 예후, 처방전, 투약, 병변
- 반드시 적절한 진료과를 안내하세요"""

# ============================================================
# [개선 3] 증상/키워드 -> 진료과 매핑 테이블
# (새 증상 테이블 반영)
#
# 증상 테이블:
# | 기침, 열, 감기           | 내과 / 이비인후과     |
# | 속쓰림, 체함, 복통       | 내과                  |
# | 설사, 혈변, 치질         | 내과 / 외과           |
# | 두통, 어지럼증           | 신경과 / 내과         |
# | 허리·목 통증             | 정형외과              |
# | 손저림, 다리 당김        | 신경과 / 정형외과     |
# | 여드름, 피부염           | 피부과                |
# | 눈 충혈, 시야 흐림       | 안과                  |
# | 귀 먹먹함, 이통, 이명    | 이비인후과            |
# | 잇몸통증, 충치           | 치과                  |
# | 생리불순, 질염           | 산부인과              |
# | 소변 시 통증, 잔뇨감     | 비뇨의학과            |
# | 가슴 두근거림, 압박      | 내과 / 응급실         |
# | 호흡곤란, 기침 지속      | 내과                  |
# | 갑상선, 당뇨, 체중변화   | 내과                  |
# | 아이 열, 감기, 피부트러블 | 소아청소년과          |
# | 사고, 골절, 찢김         | 정형외과 / 응급실     |
# | 교통사고 후 통증         | 정형외과              |
# ============================================================

DEPT_MAP = {
    # 정형외과
    '무릎': '정형외과', '허리': '정형외과', '어깨': '정형외과',
    '관절': '정형외과', '골절': '정형외과', '뼈': '정형외과',
    '발목': '정형외과', '손목': '정형외과', '척추': '정형외과',
     '찢김': '정형외과 / 응급실',
    '사고': '정형외과 / 응급실',
    # 내과 (기침·감기·소화기·만성질환 통합)
    '기침': '내과 / 이비인후과', '감기': '내과 / 이비인후과',
    '열': '내과', '소화': '내과', '속쓰림': '내과',
    '체함': '내과', '혈압': '내과', '당뇨': '내과',
    '갑상선': '내과', '체중변화': '내과', '호흡곤란': '내과',
    '위': '내과', '복통': '내과', '설사': '내과 / 외과',
    '구토': '내과', '혈변': '내과 / 외과', '치질': '내과 / 외과',
    # 신경과
    '두통': '신경과 / 내과', '어지럼': '신경과 / 내과',
    '마비': '신경과', '저림': '신경과',
    '기억력': '신경과', '머리': '신경과 / 내과',
    '손저림': '신경과 / 정형외과', '다리 당김': '신경과 / 정형외과',
    # 피부과
    '가려움': '피부과', '발진': '피부과', '두드러기': '피부과',
    '피부': '피부과', '여드름': '피부과', '피부염': '피부과',
    # 안과
    '눈': '안과', '시력': '안과', '충혈': '안과', '시야': '안과',
    # 이비인후과
    '귀': '이비인후과', '목': '이비인후과', '코': '이비인후과',
    '인후': '이비인후과', '편도': '이비인후과',
    '이통': '이비인후과', '이명': '이비인후과',
    # 치과
    '치아': '치과', '잇몸': '치과', '충치': '치과',
    # 비뇨의학과
    '소변': '비뇨의학과', '전립선': '비뇨의학과', '잔뇨': '비뇨의학과',
    # 산부인과
    '생리': '산부인과', '질염': '산부인과',
    # 소아청소년과
    '소아': '소아청소년과', '아이': '소아청소년과',
    # 내과 / 응급실 (가슴 두근거림·압박)
    '두근': '내과 / 응급실', '가슴': '내과 / 응급실',
}

def get_dept_from_text(text: str, entities: dict = None) -> str:
    """텍스트와 엔티티에서 진료과 추출"""
    if entities:
        body_part = entities.get('body_part', '')
        if body_part and body_part in DEPT_MAP:
            return DEPT_MAP[body_part]
    for keyword, dept in DEPT_MAP.items():
        if keyword in text:
            return dept
    return ''

def ensure_dept_in_answer(answer: str, dept: str) -> str:
    """답변에 진료과가 없으면 자동으로 추가"""
    if not dept:
        return answer
    if dept in answer or '과에' in answer or '과를' in answer:
        return answer
    return answer.rstrip() + f' {dept}에 가보시는 게 좋을 것 같아요.'

def generate_answer_local(text: str, context: str = '', entities: dict = None) -> dict:
    """파인튜닝 모델로 답변 생성 + 진료과 보장"""
    prompt = f'### 질문:\n{text}\n\n### 답변:\n'
    if context:
        prompt = f'참고 정보: {context}\n\n{prompt}'

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to('cuda')

    outputs = llm_model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = result.split('### 답변:')[-1].strip()

    dept = get_dept_from_text(text, entities)
    answer = ensure_dept_in_answer(answer, dept)

    return {'answer': answer, 'model': 'LG_HelloDoctor/LLM'}


def generate_answer_groq(text: str, context: str = '', entities: dict = None) -> dict:
    """Groq API로 답변 생성 (fallback) + 진료과 보장"""
    user_msg = f'참고 정보: {context}\n\n질문: {text}' if context else text

    response = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg}
        ],
        max_tokens=150,
        temperature=0.3,
    )
    answer = response.choices[0].message.content.strip()

    dept = get_dept_from_text(text, entities)
    answer = ensure_dept_in_answer(answer, dept)

    return {'answer': answer, 'model': 'groq-llama-3.3-70b'}


def generate_answer(text: str, context: str = '', confidence: float = 0.85, entities: dict = None) -> dict:
    """신뢰도 기반 라우팅"""
    if confidence >= 0.7:
        try:
            return generate_answer_local(text, context, entities)
        except Exception as e:
            print(f'로컬 오류 -> Groq fallback: {e}')
            return generate_answer_groq(text, context, entities)
    else:
        print('신뢰도 낮음 -> Groq fallback')
        return generate_answer_groq(text, context, entities)


# 테스트
print('=== 로컬 모델 ===')
r = generate_answer('무릎이 너무 아파요. 어디 가야 해요?', confidence=0.91)
print(f'모델: {r["model"]}\n답변: {r["answer"]}')

print('\n=== Groq Fallback ===')
r = generate_answer('무릎이 너무 아파요. 어디 가야 해요?', confidence=0.5)
print(f'모델: {r["model"]}\n답변: {r["answer"]}')


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 로컬 모델 ===


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


모델: LG_HelloDoctor/LLM
답변: 정형외과에 가보시는 게 좋겠어요.

=== Groq Fallback ===
신뢰도 낮음 -> Groq fallback
모델: groq-llama-3.3-70b
답변: 무릎이 아프면 관절 관련 문제일 수 있으니, 정형외과를 방문하시는 것이 좋을 것 같습니다. 의사가 정확한 진단을 해주실 수 있을 거예요. 가까운 병원에 가서 정형외과를 찾아보세요.


## Step 13 — 다중 턴 대화 (추가 질문)

In [17]:
import re

# ============================================================
# 부위별 추가 질문 설계 (새 증상 추가)
# ============================================================
FOLLOWUP_QUESTIONS_V2 = {
    '팔':   '팔이 저리시군요. 갑자기 저리신 건가요, 오래 전부터 저리셨나요?',
    '다리': '다리가 저리시군요. 갑자기 저리신 건가요, 오래 전부터 저리셨나요?',
    '손':   '손이 저리시군요. 갑자기 저리신 건가요, 오래 전부터 저리셨나요?',
    '발':   '발이 저리시군요. 갑자기 저리신 건가요, 오래 전부터 저리셨나요?',
    '무릎': '무릎이 아프시군요. 걷기가 많이 힘드세요?',
    '허리': '허리가 아프시군요. 다리까지 저리거나 당기세요?',
    '어깨': '어깨가 아프시군요. 팔을 올리기 힘드세요?',
    '머리': '머리가 아프시군요. 갑자기 생긴 통증인가요?',
    '두통': '두통이 있으시군요. 갑자기 생긴 통증인가요?',
    '가슴': '가슴이 불편하시군요. 숨쉬기가 힘드세요?',
    '배':   '배가 아프시군요. 오른쪽 아래가 아프신가요, 아니면 속이 쓰리신가요?',
    '위':   '위가 불편하시군요. 속이 쓰리거나 신물이 올라오세요?',
    '어지': '어지럼증이 있으시군요. 빙빙 도는 느낌인가요?',
    '눈':   '눈이 불편하시군요. 갑자기 안 보이시는 건가요?',
    '귀':   '귀가 불편하시군요. 갑자기 안 들리시는 건가요?',
    '목':   '목이 아프시군요. 침을 삼키기 힘드세요?',
    '피부': '피부가 불편하시군요. 갑자기 두드러기가 생긴 건가요?',
    '소변': '소변이 불편하시군요. 소변 볼 때 따갑거나 피가 나오시나요?',
    '치아': '치아가 아프시군요. 잇몸이 붓거나 피가 나오나요?',
    '기침': '기침이 있으시군요. 열도 나시나요?',
    '감기': '감기 증상이 있으시군요. 열이나 목 통증도 있으신가요?',
    '생리': '생리 관련 불편이 있으시군요. 통증이 심하신가요?',
    '아이': '아이가 아프군요. 열이 나거나 피부 트러블이 있나요?',
}

# 2턴: 답변에 따라 단일 진료과 결정 (내과로 통합)
DEPT_DECISION_V2 = {
    '팔': {
        '갑자기': ('신경과', '신경과에 가보시는 게 좋을 것 같아요. 뇌나 신경 문제일 수 있어요.'),
        '오래전': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요. 디스크나 신경 눌림일 수 있어요.'),
        'default': ('신경과', '신경과에 가보시는 게 좋을 것 같아요.'),
    },
    '다리': {
        '갑자기': ('신경과', '신경과에 가보시는 게 좋을 것 같아요. 빨리 가보시는 게 좋아요.'),
        '오래전': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        'default': ('신경과', '신경과에 가보시는 게 좋을 것 같아요.'),
    },
    '손': {
        '갑자기': ('신경과', '신경과에 가보시는 게 좋을 것 같아요.'),
        '오래전': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        'default': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
    },
    '발': {
        '갑자기': ('신경과', '신경과에 가보시는 게 좋을 것 같아요.'),
        '오래전': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        'default': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
    },
    '무릎': {
        '심함': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        '가벼움': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        'default': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
    },
    '허리': {
        '다리저림': ('신경외과', '신경외과에 가보시는 게 좋을 것 같아요. 디스크가 신경을 누르고 있을 수 있어요.'),
        '허리만': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        'default': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
    },
    '어깨': {
        '심함': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        '가벼움': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
        'default': ('정형외과', '정형외과에 가보시는 게 좋을 것 같아요.'),
    },
    '머리': {
        '갑자기': ('신경과', '신경과에 빨리 가보시는 게 좋을 것 같아요. 갑작스러운 두통은 위험할 수 있어요.'),
        '오래전': ('신경과 / 내과', '신경과나 내과에 가보시는 게 좋을 것 같아요.'),
        'default': ('신경과', '신경과에 가보시는 게 좋을 것 같아요.'),
    },
    '두통': {
        '갑자기': ('신경과', '신경과에 빨리 가보시는 게 좋을 것 같아요.'),
        '오래전': ('신경과 / 내과', '신경과나 내과에 가보시는 게 좋을 것 같아요.'),
        'default': ('신경과', '신경과에 가보시는 게 좋을 것 같아요.'),
    },
    '가슴': {
        '숨힘들': ('내과 / 응급실', '내과 또는 응급실에 빨리 가보시는 게 좋을 것 같아요.'),
        '괜찮': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
        'default': ('내과 / 응급실', '내과 또는 응급실에 가보시는 게 좋을 것 같아요.'),
    },
    '배': {
        '속쓰림': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
        '오른쪽': ('외과', '외과에 빨리 가보시는 게 좋을 것 같아요. 맹장일 수 있어요.'),
        '설사': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
        'default': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
    },
    '위': {
        '속쓰림': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
        '오래전': ('내과', '내과에서 위 검사를 받아보시는 게 좋을 것 같아요.'),
        'default': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
    },
    '어지': {
        '빙빙': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요. 이석증일 수 있어요.'),
        '캄캄': ('신경과', '신경과에 빨리 가보시는 게 좋을 것 같아요.'),
        'default': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
    },
    '눈': {
        '갑자기': ('안과', '안과에 빨리 가보시는 게 좋을 것 같아요.'),
        '오래전': ('안과', '안과에 가보시는 게 좋을 것 같아요.'),
        'default': ('안과', '안과에 가보시는 게 좋을 것 같아요.'),
    },
    '귀': {
        '갑자기': ('이비인후과', '이비인후과에 빨리 가보시는 게 좋을 것 같아요.'),
        '오래전': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
        'default': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
    },
    '목': {
        '심함': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
        '가벼움': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
        'default': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
    },
    '피부': {
        '갑자기': ('피부과', '피부과에 빨리 가보시는 게 좋을 것 같아요.'),
        '오래전': ('피부과', '피부과에 가보시는 게 좋을 것 같아요.'),
        'default': ('피부과', '피부과에 가보시는 게 좋을 것 같아요.'),
    },
    '소변': {
        '따가움': ('비뇨의학과', '비뇨의학과에 가보시는 게 좋을 것 같아요. 방광염일 수 있어요.'),
        '피':    ('비뇨의학과', '비뇨의학과에 빨리 가보시는 게 좋을 것 같아요.'),
        'default': ('비뇨의학과', '비뇨의학과에 가보시는 게 좋을 것 같아요.'),
    },
    '치아': {
        '잇몸': ('치과', '치과에 가보시는 게 좋을 것 같아요. 치주염일 수 있어요.'),
        '통증': ('치과', '치과에 가보시는 게 좋을 것 같아요.'),
        'default': ('치과', '치과에 가보시는 게 좋을 것 같아요.'),
    },
    '기침': {
        '열있음': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
        '목아픔': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
        'default': ('내과 / 이비인후과', '내과나 이비인후과에 가보시는 게 좋을 것 같아요.'),
    },
    '감기': {
        '열있음': ('내과', '내과에 가보시는 게 좋을 것 같아요.'),
        '목아픔': ('이비인후과', '이비인후과에 가보시는 게 좋을 것 같아요.'),
        'default': ('내과 / 이비인후과', '내과나 이비인후과에 가보시는 게 좋을 것 같아요.'),
    },
    '생리': {
        '심함': ('산부인과', '산부인과에 가보시는 게 좋을 것 같아요.'),
        '가벼움': ('산부인과', '산부인과에 가보시는 게 좋을 것 같아요.'),
        'default': ('산부인과', '산부인과에 가보시는 게 좋을 것 같아요.'),
    },
    '아이': {
        '열있음': ('소아청소년과', '소아청소년과에 가보시는 게 좋을 것 같아요.'),
        '피부': ('소아청소년과', '소아청소년과에 가보시는 게 좋을 것 같아요.'),
        'default': ('소아청소년과', '소아청소년과에 가보시는 게 좋을 것 같아요.'),
    },
}

# 2턴 답변 -> 키 매핑
TURN2_KEYWORD_MAP = {
    '갑자기': ['갑자기', '갑작스럽', '갑자기요', '갑자기 나타'],
    '오래전': ['오래', '예전부터', '계속', '며칠', '몇 주', '몇 달', '만성', '꽤'],
    '심함':   ['많이', '너무', '심해', '못 걷', '힘들', '매우', '엄청', '걷기 힘', '심하'],
    '가벼움': ['조금', '약간', '살짝', '가끔', '그냥', '괜찮은 편'],
    '다리저림': ['다리', '저려', '당겨', '내려가'],
    '허리만': ['허리만', '허리만요', '허리 쪽만'],
    '속쓰림': ['쓰려', '쓰린', '신물', '역류', '속이 쓰'],
    '설사': ['설사', '묽은', '물처럼'],
    '오른쪽': ['오른쪽', '오른편', '오른 쪽'],
    '빙빙':   ['빙빙', '돌아요', '빙글', '돌고'],
    '캄캄':   ['캄캄', '어두워', '안 보', '잠깐 안'],
    '숨힘들': ['숨', '호흡', '숨쉬기', '숨이 차'],
    '괜찮':   ['괜찮', '숨은 괜찮', '숨쉬기는'],
    '따가움': ['따가', '따끔', '아파요', '통증'],
    '피':     ['피', '혈뇨', '붉은'],
    '잇몸':   ['잇몸', '잇몸이', '피가 나'],
    '통증':   ['아파', '통증', '쑤셔', '욱신'],
    '열있음': ['열', '고열', '미열', '체온'],
    '목아픔': ['목', '목 아파', '목이 아'],
    '피부':   ['피부', '발진', '두드러기', '트러블'],
}

def detect_turn2_key(text: str, body_part: str) -> str:
    """2턴 답변에서 진료과 결정 키 추출"""
    dept_map = DEPT_DECISION_V2.get(body_part, {})
    available_keys = [k for k in dept_map.keys() if k != 'default']

    for key in available_keys:
        keywords = TURN2_KEYWORD_MAP.get(key, [key])
        if any(kw in text for kw in keywords):
            return key

    return 'default'


# ============================================================
# 고도화 다중 턴 대화 함수
# ============================================================
conversation_state_v2 = {}

def chat_with_followup_v2(user_input: str, session_id: str = 'default') -> dict:
    """
    고도화 다중 턴 대화:
    1턴: 증상 파악, 증상 특성 추가 질문
    2턴: 증상 특성 파악, 단일 진료과 안내
    응급: 즉시 119 안내
    """
    global conversation_state_v2

    if session_id not in conversation_state_v2:
        conversation_state_v2[session_id] = {
            'step': 1,
            'body_part': None
        }

    state = conversation_state_v2[session_id]

    # 응급 먼저 체크
    intent = classify_intent(user_input)
    if intent['intent'] == 'emergency':
        conversation_state_v2[session_id] = {'step': 1, 'body_part': None}
        return {
            'answer': '지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.',
            'intent': 'emergency',
            'department': None,
            'ready_for_c': True
        }

    # 복약·병원 검색 즉시 처리
    if intent['intent'] in ['medication_info', 'hospital_search']:
        return {
            'answer': None,
            'intent': intent['intent'],
            'department': None,
            'ready_for_c': True
        }

    # 1턴: 신체 부위 감지 -> 추가 질문
    if state['step'] == 1:
        for part, question in FOLLOWUP_QUESTIONS_V2.items():
            if part in user_input:
                state['body_part'] = part
                state['step'] = 2
                return {
                    'answer': question,
                    'intent': 'symptom_inquiry',
                    'department': None,
                    'ready_for_c': False
                }
        # 부위 못 찾으면 바로 C팀으로
        return {
            'answer': None,
            'intent': 'symptom_inquiry',
            'department': None,
            'ready_for_c': True
        }

    # 2턴: 증상 특성 파악 -> 단일 진료과 결정
    if state['step'] == 2:
        body_part = state['body_part']
        turn2_key = detect_turn2_key(user_input, body_part)

        dept_info  = DEPT_DECISION_V2.get(body_part, {})
        dept, answer = dept_info.get(
            turn2_key,
            dept_info.get('default', ('내과', '내과에 가보시는 게 좋을 것 같아요.'))
        )

        conversation_state_v2[session_id] = {'step': 1, 'body_part': None}

        return {
            'answer': answer,
            'intent': 'symptom_inquiry',
            'department': dept,
            'ready_for_c': True,
            'output_for_c': {
                'intent':     'symptom_inquiry',
                'entities':   {'body_part': body_part, 'symptom': body_part + ' 불편'},
                'query':      user_input,
                'confidence': 0.91,
                'department': dept
            }
        }


# ============================================================
# 테스트
# ============================================================
print('=' * 55)
print('시나리오 A: 팔 저림 (갑자기 -> 신경과)')
print('=' * 55)
r1 = chat_with_followup_v2('팔이 저려요', 'A')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('갑자기 저려요', 'A')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')

print()
print('=' * 55)
print('시나리오 B: 허리 통증 (다리 저림 -> 신경외과)')
print('=' * 55)
r1 = chat_with_followup_v2('허리가 아파요', 'B')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('다리까지 저리고 당겨요', 'B')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')

print()
print('=' * 55)
print('시나리오 C: 가슴 (숨힘들 -> 내과/응급실)')
print('=' * 55)
r1 = chat_with_followup_v2('가슴이 아파요', 'C')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('숨쉬기도 힘들어요', 'C')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')

print()
print('=' * 55)
print('시나리오 D: 기침 (열 있음 -> 내과)')
print('=' * 55)
r1 = chat_with_followup_v2('기침이 나요', 'D')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('열도 나요', 'D')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')

print()
print('=' * 55)
print('시나리오 E: 응급 (즉시 처리)')
print('=' * 55)
r1 = chat_with_followup_v2('가슴이 너무 아프고 숨이 안 쉬어져요', 'E')
print(f'1턴 AI:  {r1["answer"]}')
print(f'의도:    {r1["intent"]}')

print()
print('=' * 55)
print('시나리오 F: 배 통증 (오른쪽 -> 외과)')
print('=' * 55)
r1 = chat_with_followup_v2('배가 아파요', 'F')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('오른쪽 아래가 아파요', 'F')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')

print()
print('=' * 55)
print('시나리오 G: 생리불순')
print('=' * 55)
r1 = chat_with_followup_v2('생리불순이 있어요', 'G')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('통증이 심해요', 'G')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')

print()
print('=' * 55)
print('시나리오 H: 아이 열')
print('=' * 55)
r1 = chat_with_followup_v2('아이가 열이 나요', 'H')
print(f'1턴 AI:  {r1["answer"]}')
r2 = chat_with_followup_v2('열이 많이 나요', 'H')
print(f'2턴 AI:  {r2["answer"]}')
print(f'진료과:  {r2["department"]}')


시나리오 A: 팔 저림 (갑자기 -> 신경과)
1턴 AI:  팔이 저리시군요. 갑자기 저리신 건가요, 오래 전부터 저리셨나요?
2턴 AI:  신경과에 가보시는 게 좋을 것 같아요. 뇌나 신경 문제일 수 있어요.
진료과:  신경과

시나리오 B: 허리 통증 (다리 저림 -> 신경외과)
1턴 AI:  허리가 아프시군요. 다리까지 저리거나 당기세요?
2턴 AI:  신경외과에 가보시는 게 좋을 것 같아요. 디스크가 신경을 누르고 있을 수 있어요.
진료과:  신경외과

시나리오 C: 가슴 (숨힘들 -> 내과/응급실)
1턴 AI:  가슴이 불편하시군요. 숨쉬기가 힘드세요?
2턴 AI:  지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.
진료과:  None

시나리오 D: 기침 (열 있음 -> 내과)
1턴 AI:  기침이 있으시군요. 열도 나시나요?
2턴 AI:  내과에 가보시는 게 좋을 것 같아요.
진료과:  내과

시나리오 E: 응급 (즉시 처리)
1턴 AI:  지금 바로 119에 전화해 주세요. 매우 위험한 상황이에요.
의도:    emergency

시나리오 F: 배 통증 (오른쪽 -> 외과)
1턴 AI:  배가 아프시군요. 오른쪽 아래가 아프신가요, 아니면 속이 쓰리신가요?
2턴 AI:  외과에 빨리 가보시는 게 좋을 것 같아요. 맹장일 수 있어요.
진료과:  외과

시나리오 G: 생리불순
1턴 AI:  생리 관련 불편이 있으시군요. 통증이 심하신가요?
2턴 AI:  산부인과에 가보시는 게 좋을 것 같아요.
진료과:  산부인과

시나리오 H: 아이 열
1턴 AI:  아이가 아프군요. 열이 나거나 피부 트러블이 있나요?
2턴 AI:  소아청소년과에 가보시는 게 좋을 것 같아요.
진료과:  소아청소년과


## Step 15 — 모델 평가 및 성능 비교

In [18]:
import json
import time
import torch
from tqdm import tqdm

INTENT_MAX_INPUT_LENGTH = 256
INTENT_BATCH_SIZE = 16
INTENT_MAX_NEW_TOKENS = 8
EVAL_PATH = '/content/drive/MyDrive/LG_HelloDoctor/LLM/data/test.jsonl'
SAVE_DIR  = '/content/drive/MyDrive/LG_HelloDoctor/LLM/results'

# =========================================================
# 빠른 평가용 설정
# =========================================================
INTENT_BATCH_SIZE = 32              # GPU 여유 있으면 64까지 시도
SAVE_RAW_INTENT_OUTPUT = False      # raw_output 저장 최소화

# =========================================================
# 유틸
# =========================================================
def safe_div(a, b):
    return (a / b * 100) if b else 0.0

def parse_intent_from_output(result: str) -> str:
    result = (result or "").strip()
    VALID_INTENTS = ['symptom_inquiry', 'hospital_search', 'medication_info', 'emergency']

    if result in VALID_INTENTS:
        return result

    lower = result.lower().strip()
    if lower in VALID_INTENTS:
        return lower

    for intent in VALID_INTENTS:
        if intent in lower:
            return intent

    tokens = lower.split()
    if tokens and tokens[0] in VALID_INTENTS:
        return tokens[0]

    return 'symptom_inquiry'

def build_intent_prompt(text: str) -> str:
    return (
        "당신은 의료 AI 분류기입니다. 문장의 의도를 4가지 중 하나로만 답하세요.\n"
        "의도 종류:\n"
        "- symptom_inquiry: 증상 문의\n"
        "- hospital_search: 병원 검색\n"
        "- medication_info: 약 문의\n"
        "- emergency: 응급 상황\n\n"
        f"문장: {text}\n"
        "의도:"
    )

def category_to_intent(category: str) -> str:
    if 'emergency' in category or '응급' in category:
        return 'emergency'
    elif 'hospital' in category or '병원' in category:
        return 'hospital_search'
    elif 'medication' in category or '복약' in category or '약' in category:
        return 'medication_info'
    else:
        return 'symptom_inquiry'

# =========================================================
# 평가 데이터 로드
# =========================================================
eval_data = []
with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        text   = item.get('instruction', '')
        intent = item.get('intent', 'symptom_inquiry')  # ← category_to_intent 대신 intent 필드 직접 사용
        dept   = item.get('department', '')
        eval_data.append((text, intent, dept))

print(f'평가 데이터 로드 완료: {len(eval_data)}개')
if eval_data:
    print(f'샘플: {eval_data[0]}')

# =========================================================
# 의도 분류 배치
# =========================================================
def classify_intent_batch(texts, batch_size=32):
    results = []
    non_keyword_texts = []
    non_keyword_idx = []

    # 1단계: 키워드 선반영 (응급 + 복약 + 병원검색)
    pre_results = {}
    for i, text in enumerate(texts):
        # 멀티턴 마지막 발화 추출
        if '[2턴]' in text:
            last_turn = text.split('[2턴]')[-1].strip()
            negation = ['괜찮', '아니요', '오래됐', '없어요', '그냥']
            check_emergency = not any(neg in last_turn for neg in negation)
        else:
            last_turn = text.strip()
            check_emergency = True

        full_text = text + ' ' + last_turn

        # 응급 (부정 표현 없을 때만)
        if check_emergency and is_emergency(full_text):
            pre_results[i] = {
                'intent': 'emergency',
                'confidence': 0.99,
                'raw_output': 'keyword_match'
            }
        # 복약
        elif any(kw in full_text for kw in MEDICATION_KW):
            pre_results[i] = {
                'intent': 'medication_info',
                'confidence': 0.90,
                'raw_output': 'keyword_match'
            }
        # 병원 검색
        elif any(kw in full_text for kw in HOSPITAL_KW):
            pre_results[i] = {
                'intent': 'hospital_search',
                'confidence': 0.90,
                'raw_output': 'keyword_match'
            }
        else:
            non_keyword_texts.append(text)
            non_keyword_idx.append(i)

    # 2단계: 나머지만 LLM 배치 추론
    llm_results = {}
    for start_idx in tqdm(
        range(0, len(non_keyword_texts), batch_size),
        desc='[1/2] Intent Batch Inference'
    ):
        batch_texts = non_keyword_texts[start_idx:start_idx + batch_size]
        batch_idx   = non_keyword_idx[start_idx:start_idx + batch_size]
        prompts = [build_intent_prompt(t) for t in batch_texts]

        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=INTENT_MAX_INPUT_LENGTH
        )
        inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=INTENT_MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        if getattr(llm_model.config, 'is_encoder_decoder', False):
            decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        else:
            prompt_len = inputs['input_ids'].shape[1]
            gen_only = outputs[:, prompt_len:]
            decoded_outputs = tokenizer.batch_decode(gen_only, skip_special_tokens=True)

        for i, raw in zip(batch_idx, decoded_outputs):
            raw = raw.strip()
            pred_intent = parse_intent_from_output(raw)
            llm_results[i] = {
                'intent': pred_intent,
                'confidence': 1.0,
                'raw_output': raw
            }

    # 원래 순서 복원
    for i in range(len(texts)):
        if i in pre_results:
            results.append(pre_results[i])
        else:
            results.append(llm_results[i])

    return results

# =========================================================
# 전체 의도 분류
# =========================================================
all_texts = [x[0] for x in eval_data]

start_time = time.time()
intent_outputs = classify_intent_batch(all_texts, batch_size=INTENT_BATCH_SIZE)
print(f'의도 분류 완료: {len(intent_outputs)}건')

# =========================================================
# 빠른 평가 루프
# - 의도 정확도
# - 응급 감지율
# - 오답 케이스만 기록
# =========================================================
intent_correct = 0
emergency_correct = 0
emergency_total = 0

wrong_cases = []
eval_results = []

for (text, true_intent, true_dept), intent_result in tqdm(
    zip(eval_data, intent_outputs),
    total=len(eval_data),
    desc='[2/2] Fast Scoring'
):
    pred_intent = intent_result['intent']
    raw_output  = intent_result.get('raw_output', '')
    confidence  = intent_result.get('confidence', 1.0)

    intent_ok = (pred_intent == true_intent)
    if intent_ok:
        intent_correct += 1
    else:
        wrong_cases.append({
            'text': text,
            'true': true_intent,
            'pred': pred_intent,
            'raw_output': raw_output
        })

    if true_intent == 'emergency':
        emergency_total += 1
        if pred_intent == 'emergency':
            emergency_correct += 1

    row = {
        'text': text,
        'true_intent': true_intent,
        'pred_intent': pred_intent,
        'intent_ok': intent_ok,
        'confidence': confidence,
    }

    if SAVE_RAW_INTENT_OUTPUT:
        row['intent_raw_output'] = raw_output

    eval_results.append(row)

# =========================================================
# 결과 정리
# =========================================================
total = len(eval_data)
intent_acc = safe_div(intent_correct, total)
emergency_rate = safe_div(emergency_correct, emergency_total)
elapsed = time.time() - start_time

print('=' * 60)
print(f'📊 빠른 평가 결과 ({total:,}개)')
print('=' * 60)
print(f'의도 분류 정확도:  {intent_correct}/{total} = {intent_acc:.1f}%')
print(f'응급 감지율:       {emergency_correct}/{emergency_total} = {emergency_rate:.1f}%')
print(f'총 소요 시간:      {elapsed/60:.2f}분')
print(f'오답 케이스 수:    {len(wrong_cases)}개')

# =========================================================
# 저장
# =========================================================
# ── C팀용 b_output 생성 ──
b_outputs = []
for row in eval_results:
    text = row['text']

    # 신체부위 감지
    body_part = None
    for keyword, dept in DEPT_MAP.items():
        if keyword in text:
            body_part = dept
            break

    # 증상 추출 (핵심 키워드)
    symptom = None
    for keyword in DEPT_MAP.keys():
        if keyword in text:
            symptom = keyword
            break
    if not symptom:
        symptom = text[:20]  # 못 찾으면 앞부분

    b_outputs.append({
        'intent':   row['pred_intent'],
        'entities': {
            'symptom':   symptom,
            'body_part': body_part,
            'location':  body_part if body_part else None
        },
        'query':             text,
        'confidence':        1.0,
        'true_intent':       row['true_intent'],
        'true_dept':         row.get('true_dept') or None,
        'intent_raw_output': row.get('intent_raw_output', '')
    })

with open(f'{SAVE_DIR}/b_output_700.json', 'w', encoding='utf-8') as f:
    json.dump(b_outputs, f, ensure_ascii=False, indent=2)


with open(f'{SAVE_DIR}/fast_eval_results.json', 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

with open(f'{SAVE_DIR}/fast_wrong_cases.json', 'w', encoding='utf-8') as f:
    json.dump(wrong_cases, f, ensure_ascii=False, indent=2)

print(f'저장 완료: {SAVE_DIR}/fast_eval_results.json')
print(f'저장 완료: {SAVE_DIR}/fast_wrong_cases.json')
print(f'저장 완료: {SAVE_DIR}/fast_eval_results.json')
print(f'저장 완료: {SAVE_DIR}/fast_wrong_cases.json')
print(f'저장 완료: {SAVE_DIR}/b_output_700.json  ← C팀 전달용')

평가 데이터 로드 완료: 700개
샘플: ('[1턴] 무릎이 아파요\n[AI] 걷기가 많이 힘드세요?\n[2턴] 많이 힘들어요', 'symptom_inquiry', '')


[1/2] Intent Batch Inference:   0%|          | 0/12 [00:00<?, ?it/s]Both `max_new_tokens` (=8) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warning

의도 분류 완료: 700건


[2/2] Fast Scoring: 100%|██████████| 700/700 [00:00<00:00, 529966.21it/s]


📊 빠른 평가 결과 (700개)
의도 분류 정확도:  664/700 = 94.9%
응급 감지율:       135/150 = 90.0%
총 소요 시간:      0.17분
오답 케이스 수:    36개
저장 완료: /content/drive/MyDrive/LG_HelloDoctor/LLM/results/fast_eval_results.json
저장 완료: /content/drive/MyDrive/LG_HelloDoctor/LLM/results/fast_wrong_cases.json
저장 완료: /content/drive/MyDrive/LG_HelloDoctor/LLM/results/fast_eval_results.json
저장 완료: /content/drive/MyDrive/LG_HelloDoctor/LLM/results/fast_wrong_cases.json
저장 완료: /content/drive/MyDrive/LG_HelloDoctor/LLM/results/b_output_700.json  ← C팀 전달용


In [ ]:
# 코랩 새 셀에서 실행

# 1. 레포 클론
!git clone https://github.com/lg-hellovision-dx-data-school/LGHelloDoctor.git
%cd LGHelloDoctor

# 2. llm 브랜치로 이동
!git checkout llm

# 3. 파일 복사
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/B_llm_full.ipynb .
!mkdir -p data
!cp /content/drive/MyDrive/LG_HelloDoctor/LLM/data/00_all_medical_train.jsonl data/

# 4. 깃헙 사용자 정보 설정
!git config user.email "이메일 입력"
!git config user.name "이름 입력"



In [ ]:
import json

with open('/content/LGHelloDoctor/B_llm_full.ipynb', 'r') as f:
    content = f.read()

# 토큰 패턴 전부 제거
import re
# github_pat_ 또는 ghp_ 로 시작하는 토큰 제거
content = re.sub(r'github_pat_[A-Za-z0-9_]+', 'YOUR_TOKEN_HERE', content)
content = re.sub(r'ghp_[A-Za-z0-9]+', 'YOUR_TOKEN_HERE', content)

with open('/content/LGHelloDoctor/B_llm_full.ipynb', 'w') as f:
    f.write(content)

print('토큰 제거 완료!')
print('남은 토큰 패턴 확인:')
if 'github_pat_' in content or 'ghp_' in content:
    print('❌ 아직 토큰 남아있음')
else:
    print('✅ 토큰 없음')

In [ ]:
# 5. 커밋 & 푸시
!git add B_llm_full.ipynb data/00_all_medical_train.jsonl
!git commit -m "feat: 의료 LLM 파인튜닝 코드 및 학습 데이터 추가"
!git push origin llm